# Scientific ML on Zero Budget: Training Real Models with Free Colab/Kaggle GPUs
### SciPy India Conference Demonstration & Masterclass

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![Zero Budget AI](https://img.shields.io/badge/Compute_Budget-$0.00_(Free_Tier)-brightgreen.svg)](https://github.com/)
[![PyTorch 2.x](https://img.shields.io/badge/PyTorch-2.x_Supported-red.svg)](https://pytorch.org/)

This interactive notebook demonstrates how to train real scientific machine learning models within the memory and time limits of **free-tier GPUs (Nvidia T4 16GB, Tesla P100)** without spending a single dollar on commercial cloud compute.

---

## 1. Environment Diagnostics & Setup
Let's detect our available GPU hardware, examine compute capability, and configure reproducible seeds.

In [ ]:
# Setup dependencies if running directly inside Google Colab
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    print("Running on Google Colab! Mounting drive and setting up repository...")
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

from reproducibility import seed_everything
from utils.memory_profiler import MemoryTracker, print_gpu_hardware_summary

seed_everything(42, deterministic=True)
print_gpu_hardware_summary()

## 2. Scientific Problem: Spatio-Temporal Climate Forecasting
We simulate a real-world multi-station atmospheric forecasting problem with physical dynamics:
- Diurnal solar heating cycles (24h)
- Coupled thermodynamic properties (temperature vs humidity)
- Spatial covariance between neighboring weather monitoring stations
- **Zero external downloads required:** Generated deterministically on disk.

In [ ]:
from utils.data_streaming import (
    MemmapScientificDataset,
    generate_synthetic_climate_data,
    get_optimized_dataloader,
)

data_dir = Path("./sciml_data")
feat_path, targ_path = generate_synthetic_climate_data(
    output_dir=data_dir, num_samples=4000, seq_len=24, num_stations=8, num_features=4
)

# Zero-RAM memory mapped dataset
dataset = MemmapScientificDataset(
    features_path=feat_path,
    targets_path=targ_path,
    shape_x=(4000, 24, 8, 4),
    shape_y=(4000, 8),
)
dataloader = get_optimized_dataloader(dataset, batch_size=32, shuffle=True)
print(f"DataLoader initialized with {len(dataset)} scientific samples.")

## 3. Baseline Naive Training (FP32, Full Memory Load, No Checkpointing)
Notice how standard FP32 operations and dense layers consume significant memory and throughput.

In [ ]:
import time

import torch
import torch.nn as nn
from torch.optim import AdamW

from main_demo import BaselineHeavyScientificModel
from utils.training_utils import ScientificMetricsTracker

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_baseline = BaselineHeavyScientificModel().to(device)
optimizer = AdamW(model_baseline.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

tracker = MemoryTracker(device=device)
tracker.reset()
metrics = ScientificMetricsTracker()
t0 = time.perf_counter()

model_baseline.train()
for x_b, y_b in dataloader:
    x_b, y_b = x_b.to(device), y_b.to(device)
    optimizer.zero_grad()  # Naive zero_grad
    preds = model_baseline(x_b)
    loss = loss_fn(preds, y_b)
    loss.backward()
    optimizer.step()
    metrics.update(preds, y_b, loss.item(), x_b.size(0))

elapsed = time.perf_counter() - t0
_, _, peak_vram = tracker.get_gpu_ram_mb()
results_baseline = metrics.compute()
print(
    f"Baseline FP32 Epoch Time: {elapsed:.2f}s | Throughput: {len(dataset) / elapsed:.1f} samples/s"
)
if tracker.is_cuda:
    print(f"Baseline Peak VRAM: {peak_vram:.1f} MB")
print(f"Validation MSE: {results_baseline['mse']:.4f}")

## 4. Automatic Mixed Precision (AMP FP16) + GradScaler
AMP accelerates tensor core operations and cuts activation memory footprint in half.

In [ ]:
from utils.training_utils import AMPTrainer

model_amp = BaselineHeavyScientificModel().to(device)
optimizer_amp = AdamW(model_amp.parameters(), lr=1e-3)
amp_trainer = AMPTrainer(
    model=model_amp, optimizer=optimizer_amp, device=device, use_amp=True
)

tracker.reset()
metrics.reset()
t0 = time.perf_counter()

model_amp.train()
for x_b, y_b in dataloader:
    loss, preds = amp_trainer.forward_backward_step(x_b, y_b, loss_fn=loss_fn)
    metrics.update(preds, y_b, loss.item(), x_b.size(0))

elapsed_amp = time.perf_counter() - t0
_, _, peak_vram_amp = tracker.get_gpu_ram_mb()
results_amp = metrics.compute()
print(
    f"AMP FP16 Epoch Time: {elapsed_amp:.2f}s | Throughput: {len(dataset) / elapsed_amp:.1f} samples/s"
)
if tracker.is_cuda:
    print(
        f"AMP Peak VRAM: {peak_vram_amp:.1f} MB ({(1 - peak_vram_amp / peak_vram) * 100:.1f}% VRAM saved!)"
    )
print(f"Validation MSE: {results_amp['mse']:.4f}")

## 5. Gradient Accumulation (Simulating Batch 128 on 16GB GPU)
Achieve enterprise-grade mini-batch dynamics on a single free GPU by accumulating gradients across micro-steps.

In [ ]:
from utils.training_utils import GradientAccumulator

model_accum = BaselineHeavyScientificModel().to(device)
optimizer_accum = AdamW(model_accum.parameters(), lr=1e-3)
accum_steps = 4  # Micro batch 32 * 4 = Effective Batch 128
accumulator = GradientAccumulator(accumulation_steps=accum_steps)
trainer_accum = AMPTrainer(
    model=model_accum, optimizer=optimizer_accum, device=device, use_amp=True
)

total_batches = len(dataloader)
model_accum.train()
for idx, (x_b, y_b) in enumerate(dataloader):
    should_step = accumulator.should_step(idx, total_batches)
    loss, preds = trainer_accum.forward_backward_step(
        x_b,
        y_b,
        loss_fn=loss_fn,
        accumulate_grad=(not should_step),
        accumulation_steps=accum_steps,
    )

print(
    f"Successfully trained with Effective Batch Size: {32 * accum_steps} within Micro-Batch 32 VRAM!"
)

## 6. Resilient Atomic Checkpointing
Test atomic saving and state recovery to survive unexpected Colab disconnects without corrupting files.

In [ ]:
from main_demo import EfficientScientificModel
from utils.checkpoint_manager import CheckpointManager

ckpt_dir = Path("./sciml_checkpoints")
manager = CheckpointManager(checkpoint_dir=ckpt_dir, project_name="sciml_climate")

model_save = EfficientScientificModel().to(device)
opt_save = AdamW(model_save.parameters(), lr=1e-3)

# Save checkpoint
saved_file = manager.save(model_save, opt_save, epoch=1, metric_val=0.142, is_best=True)
print(f"Saved atomic checkpoint to: {saved_file}")

# Simulate recovery
model_resumed = EfficientScientificModel().to(device)
resumed_epoch, best_metric, _ = manager.load_latest(model_resumed, device=device)
print(f"Resumed successfully! Epoch: {resumed_epoch}, Metric: {best_metric:.4f}")

## 7. Efficient Model Architecture (Depthwise Separable Convolutions + SE Blocks)
Cutting parameter count by >80% and inference latency by >3x.

In [ ]:
model_eff = EfficientScientificModel().to(device)
opt_eff = AdamW(model_eff.parameters(), lr=1e-3)
trainer_eff = AMPTrainer(
    model=model_eff, optimizer=opt_eff, device=device, use_amp=True
)

t0 = time.perf_counter()
metrics.reset()
model_eff.train()
for x_b, y_b in dataloader:
    loss, preds = trainer_eff.forward_backward_step(x_b, y_b, loss_fn=loss_fn)
    metrics.update(preds, y_b, loss.item(), x_b.size(0))

elapsed_eff = time.perf_counter() - t0
res_eff = metrics.compute()
print(
    f"Efficient Model Epoch Time: {elapsed_eff:.2f}s | Throughput: {len(dataset) / elapsed_eff:.1f} samples/s"
)
print(f"Validation MSE: {res_eff['mse']:.4f}")

## 8. Benchmark Visualizations & Compute Savings
Displaying our 4-panel publication-grade benchmark comparisons.

In [ ]:
from IPython.display import Image, display

from benchmark_results import generate_benchmark_plots, print_benchmark_tables

print_benchmark_tables()
generate_benchmark_plots("benchmark_comparison.png")
display(Image("benchmark_comparison.png"))

## 9. Compute Disclosure & Reproducibility Citation
Generate academic badge and disclosure statement for publication.

In [ ]:
from IPython.display import Markdown

from reproducibility import generate_compute_badge

badge_md = generate_compute_badge(
    platform_name="Google Colab Free (T4)",
    training_hours=0.5,
    cloud_cost_saved_usd=1.53,
)
display(Markdown(badge_md))